# 工程目录介绍
上一节讲清楚了 PyAsc 的整体架构和 API 形态。本节带你看清 PyAsc 工程的整体结构与课程随附的算子样例，为下一节的环境搭建与示例运行做准备。本节学习大纲如下。
- PyAsc 工程整体目录
- Python 前端 / 后端目录解读
- 课程随附的算子样例

---


---
## 1. PyAsc 工程整体目录
PyAsc 项目的关键目录结构如下（仅展示与算子开发相关的部分）：

```text
pyasc/
├── bin                 # 工具文件（ascir-opt / ascir-translate / ascir-lsp 等）
├── docs                # 说明文档
│   ├── figures         #   文档图片
│   └── python-api      #   API 接口文档
├── include             # 后端头文件和 td 文件
│   └── ascir           #   ascir 头文件和 td 文件
├── lib                 # 后端源文件
│   ├── Dialect         #   MLIR 方言定义源文件
│   ├── TableGen        #   tablegen 扩展代码文件
│   └── Target          #   MLIR 目标代码转换源文件
├── scripts             # 相关脚本目录
├── python              # Python 前端代码
│   ├── asc             #   用户可见的 Python 包，wheel 打包以此为主
│   ├── src             #   pybind 相关代码（C++）
│   ├── test            #   Python 单元/泛化测试用例
│   └── tutorials       #   供用户参考的样例集
└── test                # 后端测试用例
```

对算子开发者最关注的是两块：

- **`python/asc/`**：你 `import asc` 时实际加载的 Python 包。
- **`python/tutorials/`**：完整可运行的算子样例。本课程已经在 `./src/tutorials/` 下提供了两个 Add 算子样例。


### 1.1 Python 前端目录：你 `import` 的是什么
PyAsc 用户接口包 `asc` 的内部结构按职责划分为几个子模块：

```text
asc/
├── _C/             # 占位目录，保存后端自动生成的 .py 文件
├── codegen/        # 解析 Python 语法树，生成后端 MLIR
├── common/         # 公共代码（Python 版本兼容等）
├── language/       # 算子 Kernel 代码所需接口（与 Ascend C 对齐）
│   ├── adv/        #   高阶 API（Matmul 等）
│   ├── basic/      #   基础 API（data_copy、add、mul …）
│   ├── core/       #   核心数据结构和枚举（GlobalTensor、TPosition …）
│   └── fwk/        #   TPipe / TQue 等框架类
├── lib/            # C++ 库的 Python 封装
│   ├── host/       #   Ascend C Host 侧接口封装
│   └── runtime/    #   acl runtime 接口封装（rt.current_stream() 等）
└── runtime/        # Python 前端的编译与运行入口
```

几个对照关系，方便你在写算子时快速找到要 `import` 的位置：

| 想做什么 | 在 PyAsc 里的入口 |
| --- | --- |
| 写一个核函数 | `import asc` → `@asc.jit` |
| 配置 KernelType / Backend / Platform | `import asc.runtime.config as config` |
| 拿到当前 Stream | `import asc.lib.runtime as rt` → `rt.current_stream()` |
| 用 `data_copy`、`add`、`mul` 等基础 API | `asc.<api>` |
| 用 Matmul 高阶 API | `asc.Matmul` |
| 创建 Global/Local Tensor | `asc.GlobalTensor()` / `asc.LocalTensor(...)` |
| 流水同步 | `asc.set_flag(asc.HardEvent.XXX, ...)` / `asc.wait_flag(...)` |

### 1.2 后端目录：ASC-IR 与 Ascend C 代码生成
如果你只是写算子，不需要修改 PyAsc 编译器本身，可以**跳过**这一节。如果你想做 PyAsc 贡献或调试编译过程，下面是要点：

```text
include/                        # 后端头文件和 td 文件定义
└── ascir/
    ├── API/                    # API Type 定义
    ├── Dialect/                # 方言定义
    │   ├── Asc/                #   Asc 方言定义和 Pass
    │   ├── EmitAsc/            #   方言转换模块定义
    │   └── Utils/              #   公共实现
    └── Target/Asc/             # Ascend C 代码生成模块
        ├── Adv/                #   高阶 API
        ├── Basic/              #   基础 API
        ├── Core/               #   核心数据结构和枚举
        ├── External/           #   MLIR 方言代码生成
        └── Fwk/                #   TQue 等框架类

lib/                            # 后端实现，结构与 include 对应
```

这部分细节在 1.2 节的架构图中已经统一介绍。本节的目的只是让你**知道它们在哪**。


---
## 2. 课程随附的算子样例
本课程在 `./src/tutorials/` 下提供了 2 个端到端可运行的 Add 算子示例：

| 示例 | 文件位置 | 功能描述 |
| --- | --- | --- |
| `01_add` | `./src/tutorials/01_add/add.py` | 手动插入同步流水的 Add 算子 |
| `02_add_framework` | `./src/tutorials/02_add_framework/add_framework.py` | 通过 Ascend C 框架自动插入流水同步的 Add 算子 |

每个样例目录下都包含两个文件：

- `README.md`：样例规格说明（OpType、输入输出、shape、format 等）。
- `<sample>.py`：算子核函数实现 + Launch 函数 + main 入口。


---

至此你已经看懂 PyAsc 的工程结构与课程随附的算子样例。下一节会带你正式动手完成 PyAsc 开发环境的搭建，并跑通第一个 PyAsc 算子，请继续学习 [开发环境依赖与安装流程](01.04_environment_and_installation.ipynb)。


---
## 课后练习
本节介绍了 PyAsc 工程目录与课程随附的算子样例。请完成以下题目：

1. （单选题）下列哪个目录中存放 `import asc` 时实际加载的 Python 包？  
    A. `pyasc/include/`  
    B. `pyasc/lib/`  
    C. `pyasc/python/asc/`  
    D. `pyasc/python/tutorials/`  

2. （单选题）PyAsc 的 Python 接口包 `asc` 下，与 Ascend C 高阶 API（如 Matmul）一一对应的子模块是？  
    A. `asc/language/adv`  
    B. `asc/language/basic`  
    C. `asc/language/core`  
    D. `asc/language/fwk`  

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.03_answer.txt
